# Transform Sprints Data

1. Read bronze `sprints` table
2. Keep only the columns required for Analytics (drop url column)
3. Standardise column names using snake_case
4. Concatenete `name.givenName` and `name.familyName` to create a new column called `driver_name` and transform values to Title Case
5. Filter out rows where `driver_id` is null
6. Remove duplicated values
8. Write the transformated data to a silver table 

In [0]:
dbutils.widgets.text('p_batch_id', '')
v_batch_id = dbutils.widgets.get('p_batch_id')

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.sprints'
silver_table = f'{catalog_name}.{silver_schema}.sprints'

## Step 1 - Read bronze `sprints` table


In [0]:
sprints_df = spark.read.table(bronze_table).filter((F.col('batch_id') == v_batch_id))

In [0]:
display(sprints_df)

## Step 2 - Keep only the columns required for Analytics (drop url column)

In [0]:
sprints_dropped_df = sprints_df.drop('url')

##Step 3 - Standardise column names using snake_case


In [0]:
sprints_df_renamed = sprints_dropped_df.withColumnsRenamed(
    {
        'driverId': 'driver_id',
        'constructorId': 'constructor_id',
        'positionText': 'finish_position_text',
        'raceName': 'race_name',
        'date': 'race_date',
        'grid': 'grid_position',
        'laps': 'completed_laps',
        'number': 'car_number',
        'position': 'finish_position'
    }
)

##Step 4 - Transform values to Title Case

In [0]:
from pyspark.sql import functions as F

In [0]:
sprints_df_titled_case = sprints_df_renamed.withColumn('race_name', F.initcap('race_name'))

## Step 5 - Filter out rows where `driver_id` is null

In [0]:
sprints_df_not_null = sprints_df_titled_case.filter(
    (
        F.col('season').isNotNull() |
        F.col('constructor_id').isNotNull() |
        F.col('round').isNotNull() |
        F.col('season').isNotNull()
    )
)
sprints_df_not_null.count()

In [0]:
sprints_df_titled = sprints_df_not_null.withColumn('race_name', F.initcap('race_name'))

## Step 6 - Remove duplicated values

In [0]:
sprints_df_final = sprints_df_titled.dropDuplicates(['driver_id', 'constructor_id', 'round', 'season'])
sprints_df_final.count()

## Step 7 - Write the transformted data to a silver table 

In [0]:
%sql
DROP TABLE formula1.silver.sprints

In [0]:
table_key = 's.driver_id == t.driver_id AND s.constructor_id == t.constructor_id AND s.round == t.round AND s.season == t.season'

write_to_silver(
    df = sprints_df_final,
    target_table = silver_table,
    table_key = table_key,
    columns_to_update = [
        'race_date', 
        'race_name',
        'grid_position', 
        'completed_laps', 
        'grid_position', 
        'car_number', 
        'points', 
        'finish_position', 
        'finish_position_text', 
        'status', 
        'ingestion_timestamp',
        'source_file',
        'batch_id'
    ]
)

In [0]:
# (
#     sprints_df_final.write
#         .format('delta')
#         .mode('overwrite')
#         .saveAsTable(silver_table)
# )

In [0]:
%sql
SELECT * FROM formula1.silver.sprints